<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/graphs_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Table of contents

1. Loading JARVIS-DFT 3D and 2D json from figshare 
2. Making pandas dataframe,
3. Properties available, keys
4. Histogram of properties
5. Compare bandgaps from different DFT methods, OptB88vdW, TBmBJ
6. Multi-property filter for 3D material-screenig example
7. Example: find all cubic materials
8. Exfoliation energy distribution for 2D materials
9. Multi-property filter for 2D dataset

In [ ]:
!pip install alignn

In [ ]:
from jarvis.core.atoms import Atoms
from jarvis.db.figshare import data,get_jid_data

atoms=Atoms.from_dict(get_jid_data(dataset='dft_3d',jid='JVASP-32')['atoms'])

In [ ]:
from  jarvis.core.graphs import Graph
g,lg=Graph.atom_dgl_multigraph(atoms)

In [ ]:
g.edata['r']

In [ ]:
import torch,math,dgl
from dgl import convert
def radius_graph_dgl(
    cart_coords=None,
    lattice_mat=None,
    cutoff=5,
    bond_tol=0.5,
    id=None,
    atol=1e-5,
):
    """Construct edge list for radius graph."""
    cart_coords = torch.tensor(cart_coords).type(
        torch.get_default_dtype()
    )
    lattice_mat = torch.tensor(atoms.lattice_mat).type(
        torch.get_default_dtype()
    )
    inv_latt = torch.linalg.inv(lattice_mat)
    frac_coords = cart_coords@inv_latt

    #frac_coords = torch.tensor(cart_coords).type(
    #    torch.get_default_dtype()
    #)
    # elements = atoms.elements
    num_atoms = cart_coords.shape[0]
    # determine how many supercells are needed for the cutoff radius
    recp = (2 * math.pi * inv_latt).T
    recp_len = torch.tensor(
        [i for i in (torch.sqrt(torch.sum(recp**2, dim=1)))]
    )
    maxr = torch.ceil((cutoff + bond_tol) * recp_len / (2 * math.pi))
    nmin = torch.floor(torch.min(frac_coords, dim=0)[0]) - maxr
    nmax = torch.ceil(torch.max(frac_coords, dim=0)[0]) + maxr
    # construct the supercell index list

    all_ranges = [
        torch.arange(x, y, dtype=torch.get_default_dtype())
        for x, y in zip(nmin, nmax)
    ]
    cell_images = torch.cartesian_prod(*all_ranges)

    # tile periodic images into X_dst
    # index id_dst into X_dst maps to atom id as id_dest % num_atoms
    X_dst = (cell_images @ lattice_mat)[:, None, :] + cart_coords
    X_dst = X_dst.reshape(-1, 3)
    distances = torch.cdist(X_dst, X_dst, p=2, compute_mode='donot_use_mm_for_euclid_dist')
    edges = torch.nonzero(distances <= cutoff, as_tuple=True)
    g = convert.graph(edges, num_nodes=X_dst.shape[0])
    #g,r = dgl.radius_graph(X_dst, cutoff,get_distances=True)
    g.ndata['coords']=X_dst
    g.edata['r']=distances = distances[edges].unsqueeze(-1)
  
    return g,num_atoms
g1,num_atoms=radius_graph_dgl(cart_coords=torch.tensor(atoms.cart_coords),lattice_mat=torch.tensor(atoms.lattice_mat))
g1.ndata['coords'].requires_grad_(True)

In [ ]:
from torch.autograd import grad
grad(torch.linalg.norm(g1.ndata['coords']),g1.ndata['coords'],create_graph=True,retain_graph=True)

In [ ]:
g1.ndata['coords'].shape

In [ ]:
g.ndata['coords']

### We have prepared a basic json file with meta-data associated with JARVIS-DFT. So, first let's import jarvis-dft metadata from the figshre repository. This has been built as a module in the jarvis-tools


In [ ]:
# !pip install jarvis-tools, and restart runtime in the notebook if you haven't installed it yet
from jarvis.db.figshare import data

dft_3d = data('dft_3d')
dft_2d = data('dft_2d')

In [ ]:
# data-size
print (len(dft_3d),len(dft_2d))

In [ ]:
# let's make a pandas dataframe from the figshare json file
# pip install pandas, if you haven't installed yet
import pandas as pd
import numpy as np

df=pd.DataFrame(dft_3d)

In [ ]:
# Keys in the json file
print (df.columns, len(df.columns))

In [ ]:
print ('% Materials with ehull <=0.05 eV/atom: ',100*len(df[df['ehull']<=0.05])/len(df))

In [ ]:
# Now let's make a histogram for some of the material-properties in the dataset
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

the_grid = GridSpec(4, 3)
plt.rcParams.update({'font.size': 18})
plt.figure(figsize=(16,14))

plt.subplot(the_grid[0, 0])
val=np.array(df['formation_energy_peratom'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(-4,3,.1))
plt.xlabel('Formation enrgies (eV/atom)')
plt.ylabel('Materials distribution')

plt.subplot(the_grid[0, 1])
val=np.array(df['optb88vdw_bandgap'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,8,.1))
plt.xlabel('OptB88vdW bandgaps (eV)')
#plt.ylabel('Materials distribution')

plt.subplot(the_grid[0, 2])
val=np.array(df['mbj_bandgap'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,8,.1))
plt.xlabel('TBmBJ bandgaps (eV)')
#plt.ylabel('Materials distribution')

plt.subplot(the_grid[1, 0])
val=np.array(df['bulk_modulus_kv'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,250,10))
plt.xlabel('Voigt-bulk modulus (GPa)')
#plt.ylabel('Materials distribution')


plt.subplot(the_grid[1, 1])
val=np.array(df['spillage'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,4,.1))
plt.xlabel('Spin-orbit spillage')
#plt.ylabel('Materials distribution')

plt.subplot(the_grid[1, 2])
val=np.array(df['slme'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,35,1))
plt.xlabel('Solar-SLME (%)')
#plt.ylabel('Materials distribution')


plt.subplot(the_grid[2, 0])
val=np.array(df['dfpt_piezo_max_dielectric'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,50,2))
#plt.ylabel('Materials distribution')
plt.xlabel('Max.Piezo.coeff(C/m2)')


plt.subplot(the_grid[2, 1])
val=np.array(df['n-powerfact'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,4000,50))
#plt.ylabel('Materials distribution')
plt.xlabel('n-Power factor (muW/(mK)2)')

plt.subplot(the_grid[2, 2])
val=np.array(df['n-Seebeck'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(-600,0,50))
#plt.ylabel('Materials distribution')
plt.xlabel('n-Seebeck coeff (muV/K)')


plt.subplot(the_grid[3, 0])
val=np.array(df['magmom_oszicar'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,20,1))
#plt.ylabel('Materials distribution')
plt.xlabel('Mag.mom (muB)')


plt.subplot(the_grid[3, 1])
val=np.array(df['encut'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(400,1500,50))
#plt.ylabel('Materials distribution')
plt.xlabel('ENCUT (eV)')



plt.subplot(the_grid[3, 2])
val=np.sqrt(np.array(df['epsx'].replace('na',np.nan).dropna().values,dtype='float'))
plt.hist(val,bins=np.arange(0,25,1))
#plt.ylabel('Materials distribution')
plt.xlabel('OptB88vdW Refractive index-x')

plt.tight_layout()

In [ ]:
# Few more properties
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

the_grid = GridSpec(4, 3,hspace=0.8,wspace=0.3)
plt.rcParams.update({'font.size': 18})
plt.figure(figsize=(16,14))

plt.subplot(the_grid[0, 0])
val=np.array(df['spg'].replace('na',np.nan).dropna().values,dtype='int')
plt.hist(val,bins=np.arange(1,231,1))
plt.xlabel('Spg. number')
plt.ylabel('Materials distribution')

plt.subplot(the_grid[0, 1])
val=np.array(df['density'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,20,.1))
plt.xlabel('Density(g/cm3)')
#plt.ylabel('Materials distribution')

plt.subplot(the_grid[0, 2])
val=np.array(df['Tc_supercon'].replace('na',np.nan).dropna().values,dtype='float')
#val=np.array(df['slme'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,35,1))
#plt.xlabel('SLME')
plt.xlabel('Tc_supercon (K)')
#plt.ylabel('Materials distribution')

plt.subplot(the_grid[1, 0])
val=np.array(df['max_efg'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,200,5))
plt.xlabel('Electric field gradient(V/m2)')
plt.ylabel('Materials distribution')


plt.subplot(the_grid[1, 1])
val=np.array(df['ehull'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,2,.1))
plt.xlabel('Energy above convex hull (eV)')
#plt.ylabel('Materials distribution')

plt.subplot(the_grid[1, 2])
crys = pd.Categorical(df.crys)
print ('Crystal systems',crys.categories)
val=np.array(crys.codes,dtype='int')
plt.hist(val,bins=np.arange(0,8,1),width=.5)
plt.xticks([0,1,2,3,4,5,6],crys.categories.values,rotation=45)
plt.xlabel('Crystal systems')
#plt.ylabel('Materials distribution')


plt.subplot(the_grid[2, 0])
val=np.array(df['max_ir_mode'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(10,2000,50))
plt.ylabel('Materials distribution')
plt.xlabel('Max.IR mode(cm-1)')


plt.subplot(the_grid[2, 1])
val=np.array(df['min_ir_mode'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(5,100,2))
#plt.ylabel('Materials distribution')
plt.xlabel('Min.IR mode(cm-1)')

plt.subplot(the_grid[2, 2])
val=np.array(df['dfpt_piezo_max_eij'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,1,.05))
#plt.ylabel('Materials distribution')
plt.xlabel('eij')


plt.subplot(the_grid[3, 0])
val=np.array(df['dfpt_piezo_max_dij'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,20,1))
plt.ylabel('Materials distribution')
plt.xlabel('Max dij')


plt.subplot(the_grid[3, 1])
val=df['ncond'].replace('na',np.nan).dropna().values
plt.hist(val,bins=np.arange(10,1e7,1e5))
#plt.ylabel('Materials distribution')
plt.xlabel('Ncond')



plt.subplot(the_grid[3, 2])
val=df['nkappa'].replace('na',np.nan).dropna().values
plt.hist(val,bins=np.arange(10,1e16,1e14))
#plt.ylabel('Materials distribution')
plt.xlabel('nkappa')
plt.tight_layout()

In [ ]:
# Let's compare OptB88vdW and TBmBJ bandgaps
df2=df[['optb88vdw_bandgap','mbj_bandgap']].replace('na',np.nan).dropna()
plt.plot(df2['mbj_bandgap'], df2['optb88vdw_bandgap'],'.')
plt.plot(df2['mbj_bandgap'], df2['mbj_bandgap'])
plt.xlim([0,10])
plt.ylim([0,10])
plt.xlabel('TBmBJ gap (eV)')
plt.ylabel('OptB88vdW gap (eV)')
plt.tight_layout()
print ('Mats. with both OptB88vdW bandgaps', len(df['optb88vdw_bandgap']))
print ('Mats. with both TBmBJ abd OptB88vdW bandgaps', len(df2['mbj_bandgap']))

### Material-screening example: Let's find materials which has Bi element, has spin-orbit spillage >= 0.5, have OptB88vdW bandgaps > 0.1, energy above convex hull < 0.1 eV/atom , have SLME >5 %

In [ ]:
df3 = df[['optb88vdw_bandgap','spillage', 'ehull', 'slme', 'atoms', 'jid']].replace('na',np.nan).dropna()

In [ ]:
from jarvis.core.atoms import Atoms
def has_element(atoms_dict = {}, my_element = 'Bi'):
    atoms = Atoms.from_dict(atoms_dict)
    elements = atoms.elements
    if my_element in elements:
        return True
    else:
        return np.nan
    
df3['el'] = df3['atoms'].apply(lambda x: has_element(atoms_dict = x, my_element = 'Bi'))
df3['formula'] = df3['atoms'].apply(lambda x: Atoms.from_dict(x).composition.reduced_formula)

In [ ]:
df4 = df3.dropna()

In [ ]:
df4[(df4['optb88vdw_bandgap'] >= .1) & (df4['spillage'] >=.5 ) & (df4['ehull'] <=.1 ) & (df4['slme'] >=5 )  ]

### Find all cubic materials

In [ ]:
df.columns

In [ ]:
# from jarvis.analysis.structure.spacegroup import Spacegroup3D
# # Takes some time because we are running spacegroup analysis on all the materials
# def crystal_system(atoms_dict={}):
#     atoms = Atoms.from_dict(atoms_dict)
#     system = Spacegroup3D(Atoms.from_dict(atoms_dict)).crystal_system
#     return system
# df['crystal_system'] = df['atoms'].apply(lambda x: crystal_system(atoms_dict = x))

In [ ]:
len(df[df['crys']=='cubic'])

In [ ]:
# We can store the screened materials in a new json file
# Note that the primitive cell can have non-equal lattice-constants but not the conventional cell
from jarvis.db.jsonutils import dumpjson
dumpjson(data = df[df['crys']=='cubic']['atoms'].values.tolist(), filename='all_cubic.json')

### Working with 2D dataset

In [ ]:
df_2d=pd.DataFrame(dft_2d)

In [ ]:
df_2d.columns

In [ ]:
val=np.array(df_2d['exfoliation_energy'].replace('na',np.nan).dropna().values,dtype='float')
plt.hist(val,bins=np.arange(0,1000,10))
#plt.ylabel('Materials distribution')
plt.xlabel('Exfoliation energy (eV)')

In [ ]:
#Let's find exfoliable 2D materials with magnetic moment>0.5 muB , optb88vdw_bandgap > 0.05 and exfoliation_energy < 200

In [ ]:
df_2d_a = df_2d[['atoms', 'optb88vdw_bandgap','exfoliation_energy', 'magmom_oszicar','jid']].replace('na',np.nan).dropna()

In [ ]:
df_2d_a['formula'] = df_2d_a['atoms'].apply(lambda x: Atoms.from_dict(x).composition.reduced_formula)

In [ ]:
df_2d_a[(df_2d_a['optb88vdw_bandgap'] >= .05) & (df_2d_a['exfoliation_energy'] <200 ) & (df_2d_a['magmom_oszicar'] >=.5  )  ]